In [0]:
%sql
-- ============================================================================
-- 1. DIMENSÃO CALENDÁRIO (dim_calendario)
-- ============================================================================
CREATE OR REPLACE TABLE workspace.gold.dim_calendario (
  id_calendario INT COMMENT 'PK: Chave substituta no formato AAAAMMDD | Domínio: 19970101 a 20261231 | Linhagem: Derivada de data_lancamento',
  data_lancamento DATE COMMENT 'Data oficial de lançamento do jogo | Domínio: Datas válidas no formato YYYY-MM-DD | Linhagem: silver.steam_games_clean.data_lancamento',
  ano_lancamento INT COMMENT 'Ano de lançamento | Domínio: 1997 a 2026 | Linhagem: YEAR(data_lancamento)',
  mes_lancamento INT COMMENT 'Mês numérico de lançamento | Domínio: 1 (Janeiro) a 12 (Dezembro) | Linhagem: MONTH(data_lancamento)',
  trimestre_lancamento STRING COMMENT 'Trimestre do ano | Domínio: Q1, Q2, Q3, Q4 | Linhagem: QUARTER(data_lancamento)',
  decada_lancamento STRING COMMENT 'Década de lançamento do título | Domínio: Anos 1990, Anos 2000, Anos 2010, Anos 2020 | Linhagem: TRUNC(YEAR/10)*10'
)
COMMENT 'Dimensão de tempo que organiza as datas de lançamento dos jogos na Steam por ano, mês, trimestre e década.';

INSERT OVERWRITE TABLE workspace.gold.dim_calendario
SELECT DISTINCT
  CAST(DATE_FORMAT(data_lancamento, 'yyyyMMdd') AS INT) AS id_calendario,
  data_lancamento,
  YEAR(data_lancamento) AS ano_lancamento,
  MONTH(data_lancamento) AS mes_lancamento,
  CONCAT('Q', QUARTER(data_lancamento)) AS trimestre_lancamento,
  CONCAT('Anos ', CAST(FLOOR(YEAR(data_lancamento) / 10) * 10 AS INT)) AS decada_lancamento
FROM workspace.silver.steam_games_clean
WHERE data_lancamento IS NOT NULL;

-- ============================================================================
-- 2. DIMENSÃO LOCALIZAÇÃO BRASIL (dim_localizacao_br)
-- ============================================================================
CREATE OR REPLACE TABLE workspace.gold.dim_localizacao_br (
  id_localizacao INT COMMENT 'PK: Identificador único do nível de localização | Domínio: 1 a 3 | Linhagem: DENSE_RANK sobre nivel_localizacao_br',
  nivel_localizacao_br STRING COMMENT 'Categoria de suporte ao Português-BR | Domínio: Dublagem Completa (Áudio + Texto), Apenas Texto / Legenda PT-BR, Sem Suporte ao Português-BR | Linhagem: silver.steam_games_clean.nivel_localizacao_br',
  tem_texto_ptbr BOOLEAN COMMENT 'Indica presença de interface ou legenda em PT-BR | Domínio: true, false | Linhagem: silver.steam_games_clean.tem_texto_ptbr',
  tem_audio_ptbr BOOLEAN COMMENT 'Indica presença de dublagem (áudio) em PT-BR | Domínio: true, false | Linhagem: silver.steam_games_clean.tem_audio_ptbr'
)
COMMENT 'Dimensão que classifica o grau de localização dos jogos para o mercado brasileiro (Português-BR).';

INSERT OVERWRITE TABLE workspace.gold.dim_localizacao_br
SELECT
  DENSE_RANK() OVER (ORDER BY nivel_localizacao_br, tem_texto_ptbr, tem_audio_ptbr) AS id_localizacao,
  nivel_localizacao_br,
  tem_texto_ptbr,
  tem_audio_ptbr
FROM (
  SELECT DISTINCT nivel_localizacao_br, tem_texto_ptbr, tem_audio_ptbr
  FROM workspace.silver.steam_games_clean
);

-- ============================================================================
-- 3. DIMENSÃO CLASSIFICAÇÃO E PERFIL DO JOGO (dim_classificacao_jogo)
-- ============================================================================
CREATE OR REPLACE TABLE workspace.gold.dim_classificacao_jogo (
  id_classificacao INT COMMENT 'PK: Identificador único da combinação de perfil | Domínio: Inteiros >= 1 | Linhagem: DENSE_RANK sobre atributos de perfil',
  genero_principal STRING COMMENT 'Gênero primário do jogo | Domínio: Action, Indie, Adventure, RPG, Strategy, Simulation, Casual, etc. | Linhagem: silver.steam_games_clean.genero_principal',
  modalidade_jogador STRING COMMENT 'Escopo de interação entre jogadores | Domínio: Multiplayer / Co-op, Exclusivo Single-player, Outros / Não Especificado | Linhagem: silver.steam_games_clean.modalidade_jogador',
  is_indie BOOLEAN COMMENT 'Flag identificadora de estúdio independente | Domínio: true, false | Linhagem: silver.steam_games_clean.is_indie',
  faixa_preco STRING COMMENT 'Agrupamento estratégico de preço em dólar | Domínio: 1. Gratuito (Free-to-Play), 2. Até US$ 10 (Econômico), 3. US$ 10.01 a US$ 30 (Intermediário), 4. Acima de US$ 30 (Premium / AAA) | Linhagem: silver.steam_games_clean.faixa_preco',
  is_free_to_play BOOLEAN COMMENT 'Indica se o jogo é gratuito | Domínio: true, false | Linhagem: silver.steam_games_clean.is_free_to_play'
)
COMMENT 'Dimensão analítica contendo o perfil mercadológico do jogo (gênero, modalidade multiplayer/single-player, porte indie e faixa de preço).';

INSERT OVERWRITE TABLE workspace.gold.dim_classificacao_jogo
SELECT
  DENSE_RANK() OVER (ORDER BY genero_principal, modalidade_jogador, is_indie, faixa_preco, is_free_to_play) AS id_classificacao,
  genero_principal,
  modalidade_jogador,
  is_indie,
  faixa_preco,
  is_free_to_play
FROM (
  SELECT DISTINCT genero_principal, modalidade_jogador, is_indie, faixa_preco, is_free_to_play
  FROM workspace.silver.steam_games_clean
);

-- ============================================================================
-- 4. DIMENSÃO DETALHES DO JOGO E ESTÚDIOS (dim_jogo_detalhes)
-- ============================================================================
CREATE OR REPLACE TABLE workspace.gold.dim_jogo_detalhes (
  id_jogo INT COMMENT 'PK: Identificador único do aplicativo na Steam (AppID) | Domínio: Inteiros > 0 | Linhagem: silver.steam_games_clean.id_jogo',
  nome_jogo STRING COMMENT 'Título comercial do jogo | Domínio: Texto livre não nulo | Linhagem: silver.steam_games_clean.nome_jogo',
  desenvolvedora STRING COMMENT 'Estúdio(s) responsável(is) pelo desenvolvimento | Domínio: Texto livre ou Não Informado | Linhagem: silver.steam_games_clean.desenvolvedora',
  publicadora STRING COMMENT 'Empresa(s) responsável(is) pela distribuição/publicação | Domínio: Texto livre ou Não Informado | Linhagem: silver.steam_games_clean.publicadora',
  idade_minima INT COMMENT 'Classificação indicativa de idade mínima | Domínio: 0 a 21 anos | Linhagem: silver.steam_games_clean.idade_minima',
  suporte_windows BOOLEAN COMMENT 'Compatibilidade com sistema Windows | Domínio: true, false | Linhagem: silver.steam_games_clean.suporte_windows',
  suporte_mac BOOLEAN COMMENT 'Compatibilidade com sistema macOS | Domínio: true, false | Linhagem: silver.steam_games_clean.suporte_mac',
  suporte_linux BOOLEAN COMMENT 'Compatibilidade com sistema Linux | Domínio: true, false | Linhagem: silver.steam_games_clean.suporte_linux',
  faixa_proprietarios_estimada STRING COMMENT 'Estimativa de volume de cópias adquiridas (SteamSpy) | Domínio: Faixas categóricas (ex: 0 - 20000, 20000 - 50000, ..., 100000000 - 200000000) | Linhagem: silver.steam_games_clean.faixa_proprietarios_estimada'
)
COMMENT 'Dimensão cadastral contendo metadados descritivos de cada jogo, seus estúdios, sistemas operacionais suportados e faixa estimada de vendas.';

INSERT OVERWRITE TABLE workspace.gold.dim_jogo_detalhes
SELECT
  id_jogo,
  nome_jogo,
  desenvolvedora,
  publicadora,
  idade_minima,
  suporte_windows,
  suporte_mac,
  suporte_linux,
  faixa_proprietarios_estimada
FROM workspace.silver.steam_games_clean;

num_affected_rows,num_inserted_rows
125851,125851


In [0]:
%sql
-- ============================================================================
-- 5. TABELA FATO CENTRAL (fato_desempenho_jogos)
-- ============================================================================
CREATE OR REPLACE TABLE workspace.gold.fato_desempenho_jogos (
  id_jogo INT COMMENT 'FK: Referência para gold.dim_jogo_detalhes | Domínio: Inteiros > 0 | Linhagem: silver.steam_games_clean.id_jogo',
  id_calendario INT COMMENT 'FK: Referência para gold.dim_calendario | Domínio: 19970101 a 20261231 | Linhagem: JOIN com gold.dim_calendario',
  id_localizacao INT COMMENT 'FK: Referência para gold.dim_localizacao_br | Domínio: 1 a 3 | Linhagem: JOIN com gold.dim_localizacao_br',
  id_classificacao INT COMMENT 'FK: Referência para gold.dim_classificacao_jogo | Domínio: Inteiros >= 1 | Linhagem: JOIN com gold.dim_classificacao_jogo',
  preco_usd DOUBLE COMMENT 'Preço de venda padrão na loja em dólares | Domínio: 0.00 a 500.00 USD | Linhagem: silver.steam_games_clean.preco_usd',
  qtd_dlc INT COMMENT 'Quantidade de conteúdos adicionais (DLCs) disponíveis | Domínio: >= 0 | Linhagem: silver.steam_games_clean.qtd_dlc',
  qtd_idiomas_suportados INT COMMENT 'Total de idiomas suportados pelo título | Domínio: >= 0 | Linhagem: silver.steam_games_clean.qtd_idiomas_suportados',
  reviews_positivos INT COMMENT 'Total de avaliações positivas recebidas | Domínio: >= 0 | Linhagem: silver.steam_games_clean.reviews_positivos',
  reviews_negativos INT COMMENT 'Total de avaliações negativas recebidas | Domínio: >= 0 | Linhagem: silver.steam_games_clean.reviews_negativos',
  total_reviews INT COMMENT 'Soma total de avaliações (positivas + negativas) | Domínio: >= 0 | Linhagem: reviews_positivos + reviews_negativos',
  taxa_aprovacao_pct DOUBLE COMMENT 'Percentual de avaliações positivas sobre o total | Domínio: 0.00 a 100.00 (ou NULL se sem avaliações) | Linhagem: (reviews_positivos / total_reviews) * 100',
  pico_jogadores_ccu INT COMMENT 'Pico de jogadores simultâneos (Concurrent Users) | Domínio: >= 0 | Linhagem: silver.steam_games_clean.pico_jogadores_ccu',
  tempo_medio_jogo_min INT COMMENT 'Tempo médio histórico jogado pelos usuários em minutos | Domínio: >= 0 | Linhagem: silver.steam_games_clean.tempo_medio_jogo_min',
  mediana_tempo_jogo_min INT COMMENT 'Mediana do tempo jogado pelos usuários em minutos | Domínio: >= 0 | Linhagem: silver.steam_games_clean.mediana_tempo_jogo_min'
)
COMMENT 'Tabela Fato central do Esquema Estrela contendo as métricas quantitativas de preço, engajamento, retenção e aprovação dos jogos na Steam.';

INSERT OVERWRITE TABLE workspace.gold.fato_desempenho_jogos
SELECT
  s.id_jogo,
  c.id_calendario,
  l.id_localizacao,
  cl.id_classificacao,
  s.preco_usd,
  s.qtd_dlc,
  s.qtd_idiomas_suportados,
  s.reviews_positivos,
  s.reviews_negativos,
  s.total_reviews,
  s.taxa_aprovacao_pct,
  s.pico_jogadores_ccu,
  s.tempo_medio_jogo_min,
  s.mediana_tempo_jogo_min
FROM workspace.silver.steam_games_clean s
INNER JOIN workspace.gold.dim_calendario c
  ON s.data_lancamento = c.data_lancamento
INNER JOIN workspace.gold.dim_localizacao_br l
  ON s.nivel_localizacao_br = l.nivel_localizacao_br
 AND s.tem_texto_ptbr = l.tem_texto_ptbr
 AND s.tem_audio_ptbr = l.tem_audio_ptbr
INNER JOIN workspace.gold.dim_classificacao_jogo cl
  ON s.genero_principal = cl.genero_principal
 AND s.modalidade_jogador = cl.modalidade_jogador
 AND s.is_indie = cl.is_indie
 AND s.faixa_preco = cl.faixa_preco
 AND s.is_free_to_play = cl.is_free_to_play;

num_affected_rows,num_inserted_rows
125851,125851


In [0]:
%sql
SELECT 'fato_desempenho_jogos' AS tabela, COUNT(*) AS total_registros FROM workspace.gold.fato_desempenho_jogos
UNION ALL
SELECT 'dim_jogo_detalhes', COUNT(*) FROM workspace.gold.dim_jogo_detalhes
UNION ALL
SELECT 'dim_calendario', COUNT(*) FROM workspace.gold.dim_calendario
UNION ALL
SELECT 'dim_classificacao_jogo', COUNT(*) FROM workspace.gold.dim_classificacao_jogo
UNION ALL
SELECT 'dim_localizacao_br', COUNT(*) FROM workspace.gold.dim_localizacao_br;

tabela,total_registros
fato_desempenho_jogos,125851
dim_jogo_detalhes,125851
dim_calendario,5141
dim_classificacao_jogo,274
dim_localizacao_br,3
